# Análisis de la Ocupación Estacional en Hoteles

## I. Exploración Inicial

**· Objetivos del proyecto:** 

1) Analizar el comportamiento histórico de las reservas hoteleras para responder a las preguntas de negocio clave del equipo de *Revenue*.

2) Identificar patrones de estacionalidad, factores críticos en las cancelaciones, rendimiento de los canales de venta y el valor de los segmentos de clientes, todo ello en aras de optimizar las inversiones de marketing y ajustar las políticas comerciales de la compañía.

In [1]:
import pandas as pd

df = pd.read_csv('hotel_bookings.csv')

# Número de filas y columnas
df.shape 

# Primeras filas
df.head() 

# Tipos de datos y nulos
df.info() 

# Estadísticas básicas de columnas numéricas
df.describe() 

# Lista de columnas
df.columns.tolist() 

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent',
 'company',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'reservation_status',
 'reservation_status_date']

## II. Estructura y Calidad de Datos

**· Dimensiones del dataset:** 119.390 filas × 32 columnas.

**· Análisis de valores nulos detectados:**

| Columna | Valores no nulos | Nulos | % nulos aprox. |
| :--- | :--- | :--- | :--- |
| **children** | 119.386 | 4 | ~0% |
| **country** | 118.902 | 488 | ~0,4% |
| **agent** | 103.050 | 16.340 | ~13,7% |
| **company** | 6.797 | 112.593 | ~94,3% |

----

**· Observaciones y decisiones de limpieza a tomar:**

* **children:** Solo 4 nulos. Se puede imputar con 0 (asumiendo que la ausencia significa "sin niños") o eliminar esas filas, ya que el impacto es mínimo.
* **country:** ~488 nulos (0,4%), poco relevante para el análisis global. Se pueden mantener como "Desconocido" o eliminarse, según convenga para los análisis por país.
* **agent:** ~13,7% de nulos. Probablemente indica reservas sin agente intermediario (venta directa). Se podría imputar con un valor como 0 o "Sin agente" en lugar de tratarlo como dato faltante real.
* **company:** ~94,3% de nulos. Es esperable, ya que la mayoría de reservas son de particulares, no de empresas. No se eliminará la columna, pero no será una variable central salvo para segmentar reservas corporativas vs. particulares.

----

**· Tipos de datos a revisar:**

1) **Fechas de llegada:** Están repartidas en tres columnas separadas (`arrival_date_year`, `arrival_date_month`, `arrival_date_day_of_month`), por lo que habrá que combinarlas en una sola columna de tipo fecha para poder analizar la estacionalidad correctamente.
2) **Estado de reserva:** `reservation_status_date` aparece como tipo texto (str), por lo que habrá que convertirla a formato fecha.
3) **Identificadores numéricos:** `agent` y `company` están como float64 aunque conceptualmente son identificadores (categóricos); esto es coherente con que contienen nulos (NaN obliga a usar float).

----

**· Mapeo de columnas clave para las preguntas de negocio:**

* **Estacionalidad:** `arrival_date_year`, `arrival_date_month`, `arrival_date_week_number`.
* **Cancelaciones:** `is_canceled`, `reservation_status`, `lead_time`.
* **Canales de venta:** `distribution_channel`, `market_segment`, `agent`.
* **Segmentos de cliente:** `country`, `customer_type`, `adr`.

----

**· Próximos pasos del proceso de limpieza:**

1) Crear la columna `arrival_date` combinando año, mes y día.
2) Convertir `reservation_status_date` a tipo fecha.
3) Tratar los nulos en `children`, `country` y `agent` según los criterios descritos arriba.
4) Revisar valores extremos/atípicos en `adr` (por ejemplo, valores 0 o negativos).

In [2]:
# Revisión de cancelaciones
print(df['is_canceled'].value_counts())
print(df['is_canceled'].value_counts(normalize=True) * 100)

# Revisión de canal de distribución
print(df['distribution_channel'].value_counts())

# Revisión de segmento de mercado
print(df['market_segment'].value_counts())

# Revisión de ADR (precio diario)
print(df['adr'].describe())

# Valores extremos en ADR (negativos o muy altos)
print(df[df['adr'] <= 0]['adr'].count())
print(df[df['adr'] > 1000]['adr'].count())
df[df['adr'] > 1000][['hotel', 'adr', 'reservation_status']]

# Revisión de distribución geográfica (Top 10 países)
df['country'].value_counts().head(10)


is_canceled
0    75166
1    44224
Name: count, dtype: int64
is_canceled
0    62.958372
1    37.041628
Name: proportion, dtype: float64
distribution_channel
TA/TO        97870
Direct       14645
Corporate     6677
GDS            193
Undefined        5
Name: count, dtype: int64
market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Undefined            2
Name: count, dtype: int64
count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64
1960
1


country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
ITA     3766
IRL     3375
BEL     2342
BRA     2224
NLD     2104
Name: count, dtype: int64

## III. Revisión de Valores en Columnas Clave - Distribución y Outliers

**· Análisis descriptivo de variables críticas:**

1) **Cancelaciones (`is_canceled`):**
   * **No canceladas (0):** 75.166 reservas (~62,96%)
   * **Canceladas (1):** 44.224 reservas (~37,04%)
   * *Nota:* Más de un tercio de las reservas terminan cancelándose, lo que confirma que es un tema crítico para el análisis de Revenue.

2) **Canal de distribución (`distribution_channel`):**
   * **TA/TO (Agencias/Touroperadores):** 97.870 reservas (~82% del total, dominando ampliamente).
   * **Direct (Directo):** 14.645 reservas.
   * **Corporate:** 6.677 reservas.
   * **GDS:** 193 reservas.
   * **Undefined:** 5 reservas (volumen insignificante).

3) **Segmento de mercado (`market_segment`):**
   * **Online TA:** 56.477 reservas (el segmento más importante con diferencia).
   * **Offline TA/TO:** 24.219 reservas.
   * **Groups:** 19.811 reservas.
   * **Direct:** 12.606 reservas.
   * **Corporate:** 5.295 reservas.
   * **Complementary:** 743 reservas.
   * **Aviation:** 237 reservas.
   * **Undefined:** 2 reservas.

4) **Distribución geográfica (`country` - Top 10):**
   * Portugal (PRT) lidera claramente con 48.590 reservas, seguido de Reino Unido (GBR, 12.129), Francia (FRA, 10.415), España (ESP, 8.568), Alemania (DEU, 7.287), Italia (ITA, 3.766), Irlanda (IRL, 3.375), Bélgica (BEL, 2.342), Brasil (BRA, 2.224) y Países Bajos (NLD, 2.104).

5) **Métrica de ingresos (`adr` - Precio Diario Promedio):**
   * **Valor máximo registrado:** 5400.0
   * **Registros con adr <= 0:** 1.960 (probablemente reservas gratuitas, de canje de puntos, o complementarias).
   * **Registros con adr > 1000:** Solo 1, correspondiente a una reserva en "City Hotel" con `adr = 5400` y `reservation_status = Canceled`. Este es un claro outlier.

----

**· Decisiones de limpieza derivadas de la revisión:**

* **Eliminación de residuales:** Eliminar o agrupar los registros "Undefined" en `distribution_channel` (5) y `market_segment` (2), ya que su volumen es insignificante.
* **Tratamiento de outliers:** Aislar o eliminar el outlier de `adr = 5400` (City Hotel, cancelada) para evitar que distorsione las medias y métricas de ingresos globales.
* **Segmentación de tarifas cero:** Mantener identificados los 1.960 registros con `adr <= 0`. No son necesariamente errores, pero conviene excluirlos o tratarlos por separado al calcular el ADR promedio real de la compañía.

## IV. Limpieza y Transformación de Datos

A partir de las decisiones tomadas en las secciones anteriores, se aplican las siguientes transformaciones:

1) Creación de la columna `arrival_date` combinando año, mes y día de llegada.
2) Conversión de `reservation_status_date` a tipo fecha.
3) Tratamiento de nulos: `children` (imputado a 0), `country` (marcado como "Unknown"), `agent` (marcado como 0 = sin agente).
4) Eliminación de registros "Undefined" en `distribution_channel` y `market_segment`.
5) Eliminación del outlier de `adr = 5400`.
6) Creación de una columna auxiliar `is_zero_adr` para identificar reservas con `adr <= 0`, sin eliminarlas del dataset.

In [3]:
import numpy as np

# 1. Crear columna de fecha de llegada
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df['arrival_date_month_num'] = df['arrival_date_month'].map(month_map)

df['arrival_date'] = pd.to_datetime(
    dict(year=df['arrival_date_year'],
         month=df['arrival_date_month_num'],
         day=df['arrival_date_day_of_month'])
)

df = df.drop(columns=['arrival_date_month_num'])

# 2. Convertir reservation_status_date a fecha
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])

# 3. Tratar nulos
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)

# 4. Eliminar registros "Undefined"
df = df[df['distribution_channel'] != 'Undefined']
df = df[df['market_segment'] != 'Undefined']

# 5. Eliminar outlier de ADR
df = df[df['adr'] != 5400]

# 6. Marcar reservas con ADR <= 0
df['is_zero_adr'] = df['adr'] <= 0

# Verificación final
df.info()
df.shape

<class 'pandas.DataFrame'>
Index: 119384 entries, 0 to 119389
Data columns (total 34 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119384 non-null  str           
 1   is_canceled                     119384 non-null  int64         
 2   lead_time                       119384 non-null  int64         
 3   arrival_date_year               119384 non-null  int64         
 4   arrival_date_month              119384 non-null  str           
 5   arrival_date_week_number        119384 non-null  int64         
 6   arrival_date_day_of_month       119384 non-null  int64         
 7   stays_in_weekend_nights         119384 non-null  int64         
 8   stays_in_week_nights            119384 non-null  int64         
 9   adults                          119384 non-null  int64         
 10  children                        119384 non-null  float64       
 11  bab

(119384, 34)

**· Resultado de la limpieza:**

Tras aplicar las transformaciones anteriores, el dataset queda con 119.384 registros y 34 columnas (se eliminaron 6 registros: los valores "Undefined" en `distribution_channel` y `market_segment`, y el outlier de `adr = 5400`). No quedan valores nulos en las columnas tratadas, y se añadieron las columnas auxiliares `arrival_date` (fecha de llegada unificada) e `is_zero_adr` (marca de reservas con tarifa cero).

El dataset está ahora listo para ser cargado en PostgreSQL y consultado mediante SQL para responder a las preguntas de negocio definidas en el briefing.

In [4]:
df.to_csv('hotel_bookings_cleaned.csv', index=False)